# FSAKE Checkpoint Forensics

This notebook audits Hugging Face checkpoints for a specific experiment and helps explain large validation-vs-test gaps.

What it does:
- Lists files for an experiment on HF.
- Downloads `checkpoint.pth.tar` and `model_best.pth.tar` (if present).
- Prints metadata (`iteration`, `val_acc`, `best_val_acc`) and weight fingerprints.
- Verifies whether `model_best` and `checkpoint` are actually different.
- Optionally runs `eval.py` on the downloaded checkpoints if the local FSAKE repo/data are available.

Security note:
- The token is read from environment variable or prompt at runtime.
- The token is not stored in this notebook file.

In [ ]:
# Setup
import os
import json
import hashlib
from pathlib import Path
from getpass import getpass

import torch
from huggingface_hub import HfApi, hf_hub_download, login

print('Imports OK')

In [ ]:
# Config
HF_REPO_ID = 'alkav/fsake-checkpoints'
EXP_NAME = 'D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222'  # change if needed

# Prefer environment variable; fallback to secure prompt
HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
if not HF_TOKEN:
    HF_TOKEN = getpass('Enter HF token (input hidden): ').strip()

if not HF_TOKEN:
    raise RuntimeError('No HF token provided.')

login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi()

AUDIT_DIR = Path('./hf_checkpoint_audit') / EXP_NAME
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print('Repo:', HF_REPO_ID)
print('Experiment:', EXP_NAME)
print('Audit dir:', AUDIT_DIR.resolve())

In [ ]:
# List files for this experiment in HF
all_files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type='model')
exp_files = sorted([f for f in all_files if f.startswith(EXP_NAME + '/')])

print(f'Total files in repo: {len(all_files)}')
print(f'Files under {EXP_NAME}:')
for f in exp_files:
    print(' -', f)

needed = [
    f'{EXP_NAME}/checkpoint.pth.tar',
    f'{EXP_NAME}/model_best.pth.tar',
    f'{EXP_NAME}/best_log.pth.tar',
]

print('\nPresence check:')
for f in needed:
    print(f'  {f}:', 'YES' if f in all_files else 'NO')

In [ ]:
# Download checkpoints (if they exist)
local_paths = {}
for name in ['checkpoint.pth.tar', 'model_best.pth.tar', 'best_log.pth.tar']:
    remote_path = f'{EXP_NAME}/{name}'
    try:
        src = hf_hub_download(repo_id=HF_REPO_ID, filename=remote_path, repo_type='model')
        dst = AUDIT_DIR / name
        Path(dst).write_bytes(Path(src).read_bytes())
        local_paths[name] = str(dst)
        print(f'OK: {name} -> {dst}')
    except Exception as e:
        print(f'MISSING: {name} ({e})')

print('\nDownloaded files:')
for k, v in local_paths.items():
    print(' -', k, '=>', v)

In [ ]:
# Inspect checkpoint payloads and compare fingerprints
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def tensor_fingerprint(state_dict, max_tensors=8):
    # Lightweight deterministic fingerprint of the first tensors by key order
    keys = sorted(state_dict.keys())[:max_tensors]
    h = hashlib.sha256()
    for k in keys:
        t = state_dict[k].detach().cpu().contiguous()
        h.update(k.encode('utf-8'))
        h.update(str(tuple(t.shape)).encode('utf-8'))
        h.update(t.numpy().tobytes())
    return h.hexdigest(), keys

def summarize_ckpt(path):
    data = torch.load(path, map_location='cpu', weights_only=False)
    summary = {
        'path': path,
        'size_mb': round(Path(path).stat().st_size / (1024 * 1024), 2),
        'sha256': sha256_file(path),
        'keys': sorted(list(data.keys())),
        'iteration': data.get('iteration', None),
        'val_acc': data.get('val_acc', None),
        'best_val_acc': data.get('best_val_acc', None),
    }

    enc_sd = data.get('enc_module_state_dict', {})
    unet_sd = data.get('unet_module_state_dict', {})
    if enc_sd:
        summary['enc_fp'], summary['enc_fp_keys'] = tensor_fingerprint(enc_sd)
    if unet_sd:
        summary['unet_fp'], summary['unet_fp_keys'] = tensor_fingerprint(unet_sd)
    return summary

summaries = {}
for n in ['checkpoint.pth.tar', 'model_best.pth.tar']:
    if n in local_paths:
        summaries[n] = summarize_ckpt(local_paths[n])
        print('\n===', n, '===')
        for k in ['path', 'size_mb', 'sha256', 'iteration', 'val_acc', 'best_val_acc']:
            print(f'{k}:', summaries[n].get(k))

if 'checkpoint.pth.tar' in summaries and 'model_best.pth.tar' in summaries:
    same_file = summaries['checkpoint.pth.tar']['sha256'] == summaries['model_best.pth.tar']['sha256']
    same_enc = summaries['checkpoint.pth.tar'].get('enc_fp') == summaries['model_best.pth.tar'].get('enc_fp')
    same_unet = summaries['checkpoint.pth.tar'].get('unet_fp') == summaries['model_best.pth.tar'].get('unet_fp')

    print('\n=== checkpoint vs model_best comparison ===')
    print('Identical file bytes:', same_file)
    print('Same encoder weights fingerprint:', same_enc)
    print('Same UNet weights fingerprint:', same_unet)
    if same_file:
        print('WARNING: model_best and checkpoint are identical bytes.')
    elif same_enc and same_unet:
        print('WARNING: Files differ but effective model weights appear identical.')

In [ ]:
# Optional: recent repo commits (manual cross-check with HF UI)
commits = api.list_repo_commits(repo_id=HF_REPO_ID, repo_type='model')
print('Recent commits:')
for c in commits[:15]:
    print(f"- {c.commit_id[:8]} | {c.created_at} | {c.title}")

print('\nTip: Open the HF Files/History page and verify whether recent uploads for this EXP_NAME')
print('updated only checkpoint.pth.tar or also model_best.pth.tar.')

In [ ]:
# Optional: run eval.py with explicit args if local FSAKE repo and dataset exist
import subprocess
import re
import sys

FSAKE_DIR_CANDIDATES = [
    Path('/content/FSAKE'),
    Path('E:/projects/few_shot_gnn/fsake_repo_clone'),
]

EVAL_ARGS_BASELINE = [
    '--dataset', 'mini',
    '--num_ways', '5',
    '--num_shots', '1',
    '--transductive', 'True',
    '--pool_mode', 'support',
    '--unet_mode', 'addold',
    '--seed', '222',
]

acc_re = re.compile(r"evaluation: total_count=\d+, accuracy: mean=([0-9.]+)%")

def run_eval_if_possible(extra_args=None):
    extra_args = extra_args or []
    fsake_dir = next((p for p in FSAKE_DIR_CANDIDATES if (p / 'eval.py').exists()), None)
    if fsake_dir is None:
        print('No local FSAKE repo found. Skipping eval run.')
        return None

    cmd = [sys.executable, 'eval.py'] + EVAL_ARGS_BASELINE + extra_args
    print('Running in', fsake_dir)
    print('Command:', ' '.join(cmd))
    proc = subprocess.run(cmd, cwd=str(fsake_dir), capture_output=True, text=True)
    out = proc.stdout + '\n' + proc.stderr
    print(out[-4000:])

    m = acc_re.search(out)
    acc = float(m.group(1)) if m else None
    print('Return code:', proc.returncode, '| Parsed mean acc:', acc)
    return {'rc': proc.returncode, 'acc': acc, 'log': out}

baseline_result = run_eval_if_possible([])
lmt_result = run_eval_if_possible(['--interaction_block', 'lmt', '--mediator_tokens', '16', '--mediator_layers', '3', '--mediator_heads', '8'])

In [ ]:
# Final structured verdict scaffold
verdict = {
    'repo': HF_REPO_ID,
    'experiment': EXP_NAME,
    'has_checkpoint': 'checkpoint.pth.tar' in local_paths,
    'has_model_best': 'model_best.pth.tar' in local_paths,
    'checkpoint_iteration': summaries.get('checkpoint.pth.tar', {}).get('iteration'),
    'checkpoint_best_val_acc': summaries.get('checkpoint.pth.tar', {}).get('best_val_acc'),
    'model_best_iteration': summaries.get('model_best.pth.tar', {}).get('iteration'),
    'model_best_best_val_acc': summaries.get('model_best.pth.tar', {}).get('best_val_acc'),
    'checkpoint_sha256': summaries.get('checkpoint.pth.tar', {}).get('sha256'),
    'model_best_sha256': summaries.get('model_best.pth.tar', {}).get('sha256'),
}

print(json.dumps(verdict, indent=2))

if verdict['has_checkpoint'] and not verdict['has_model_best']:
    print('\nLikely issue: eval expects model_best.pth.tar but only checkpoint.pth.tar is present.')
elif verdict['has_checkpoint'] and verdict['has_model_best']:
    if verdict['checkpoint_sha256'] == verdict['model_best_sha256']:
        print('\nLikely issue: model_best and checkpoint are byte-identical; best-model selection may be ineffective or overwritten.')
    else:
        print('\nBoth checkpoint and model_best exist and differ. Next step: verify dataset/protocol consistency and run matched val/test from same loaded state.')
else:
    print('\nNo usable checkpoint found for this experiment path.')

In [ ]:
# Multi-experiment audit config (baseline + LMT)
EXPERIMENTS = [
    {
        'label': 'baseline',
        'exp_name': 'D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_SEED-222',
        'eval_args': [
            '--dataset', 'mini',
            '--num_ways', '5',
            '--num_shots', '1',
            '--transductive', 'True',
            '--pool_mode', 'support',
            '--unet_mode', 'addold',
            '--seed', '222',
        ],
    },
    {
        'label': 'lmt',
        'exp_name': 'D-mini_N-5_K-1_Q-5_B-40_T-True_P-support_Un-addold_IB-lmt_M16L3H8_SEED-222',
        'eval_args': [
            '--dataset', 'mini',
            '--num_ways', '5',
            '--num_shots', '1',
            '--transductive', 'True',
            '--pool_mode', 'support',
            '--unet_mode', 'addold',
            '--seed', '222',
            '--interaction_block', 'lmt',
            '--mediator_tokens', '16',
            '--mediator_layers', '3',
            '--mediator_heads', '8',
        ],
    },
]

print('Configured experiments:')
for e in EXPERIMENTS:
    print('-', e['label'], '=>', e['exp_name'])

In [ ]:
# Side-by-side audit for all configured experiments
from dataclasses import dataclass

@dataclass
class CkptSummary:
    exists: bool
    path: str = None
    size_mb: float = None
    sha256: str = None
    iteration: int = None
    val_acc: float = None
    best_val_acc: float = None


def download_if_exists(exp_name, filename):
    remote = f'{exp_name}/{filename}'
    try:
        src = hf_hub_download(repo_id=HF_REPO_ID, filename=remote, repo_type='model')
        dst_dir = Path('./hf_checkpoint_audit') / exp_name
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst = dst_dir / filename
        Path(dst).write_bytes(Path(src).read_bytes())
        return str(dst)
    except Exception:
        return None


def summarize_path(path):
    if not path or not Path(path).exists():
        return CkptSummary(False)
    obj = torch.load(path, map_location='cpu', weights_only=False)
    return CkptSummary(
        exists=True,
        path=path,
        size_mb=round(Path(path).stat().st_size / (1024 * 1024), 2),
        sha256=sha256_file(path),
        iteration=obj.get('iteration'),
        val_acc=obj.get('val_acc'),
        best_val_acc=obj.get('best_val_acc'),
    )


multi_audit = {}
for exp in EXPERIMENTS:
    exp_name = exp['exp_name']
    ckpt_path = download_if_exists(exp_name, 'checkpoint.pth.tar')
    best_path = download_if_exists(exp_name, 'model_best.pth.tar')

    ckpt = summarize_path(ckpt_path)
    best = summarize_path(best_path)

    multi_audit[exp_name] = {
        'checkpoint': ckpt,
        'model_best': best,
    }


def fmt_float(v):
    return 'None' if v is None else f'{v:.6f}'

print('\n=== CHECKPOINT AUDIT TABLE ===')
print('label | file | exists | iter | val_acc | best_val_acc | size_mb | sha256_prefix')
print('-' * 120)
for exp in EXPERIMENTS:
    label = exp['label']
    exp_name = exp['exp_name']
    for fname in ['checkpoint', 'model_best']:
        s = multi_audit[exp_name][fname]
        sha_pref = (s.sha256[:12] if s.sha256 else 'None')
        print(f"{label:8} | {fname:10} | {str(s.exists):6} | {str(s.iteration):>6} | {fmt_float(s.val_acc):>10} | {fmt_float(s.best_val_acc):>12} | {str(s.size_mb):>7} | {sha_pref}")

print('\n=== INTRA-EXPERIMENT COMPARISON ===')
for exp in EXPERIMENTS:
    exp_name = exp['exp_name']
    c = multi_audit[exp_name]['checkpoint']
    b = multi_audit[exp_name]['model_best']
    print(f"\n{exp['label']} -> {exp_name}")
    if not c.exists or not b.exists:
        print('  Missing one of checkpoint/model_best')
        continue
    print('  same_bytes:', c.sha256 == b.sha256)
    print('  checkpoint_iter:', c.iteration, '| model_best_iter:', b.iteration)
    print('  checkpoint_best_val_acc:', c.best_val_acc, '| model_best_best_val_acc:', b.best_val_acc)

In [ ]:
# Evaluation phase in this notebook (baseline + LMT)
# This cell runs eval.py for each configured experiment and parses mean accuracy.

import shutil
import subprocess
import re
import sys
import hashlib

ACC_RE = re.compile(r"evaluation: total_count=\d+, accuracy: mean=([0-9.]+)%, std=([0-9.]+)%, ci95=([0-9.]+)%")


def ensure_fsake_runtime_deps():
    """Install the same core runtime deps used in fsake_colab_guide for evaluation."""
    # 1) tensorboardX (required by torchtools/tt/logger.py)
    try:
        import tensorboardX  # noqa: F401
    except ImportError:
        print('Installing missing dependency: tensorboardX')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tensorboardX'], check=True)

    # 2) torch_geometric + companion wheels (guide-style install)
    try:
        import torch_geometric  # noqa: F401
    except ImportError:
        print('Installing missing dependency: torch_geometric')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)

    # Companion ops wheels from data.pyg.org matching torch/cuda, as in guide
    import torch
    torch_ver = torch.__version__.split('+')[0]
    cuda_full = (torch.version.cuda or '').replace('.', '')
    cuda_tag = f'cu{cuda_full}' if cuda_full else 'cpu'
    pyg_url = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html'
    print('PyG wheel index:', pyg_url)

    # best-effort install (already-installed packages will be skipped)
    try:
        subprocess.run(
            [
                sys.executable, '-m', 'pip', 'install', '-q',
                'pyg_lib', 'torch_scatter', 'torch_sparse', 'torch_cluster', 'torch_spline_conv',
                '-f', pyg_url,
            ],
            check=True,
        )
    except subprocess.CalledProcessError:
        # Fallback for environments where pyg_lib is not published for current combo
        subprocess.run(
            [
                sys.executable, '-m', 'pip', 'install', '-q',
                'torch_scatter', 'torch_sparse', 'torch_cluster', 'torch_spline_conv',
                '-f', pyg_url,
            ],
            check=True,
        )


ensure_fsake_runtime_deps()

# Robust repo discovery for Colab/local
FSAKE_DIR_CANDIDATES = [
    Path('/content/FSAKE'),
    Path('/content/few_shot_article/fsake_repo_clone'),
    Path('/content/drive/MyDrive/FSAKE'),
    Path('/content/few_shot_gnn/FSAKE'),
    Path('/content/few_shot_gnn/fsake_repo_clone'),
    Path('/content/fsake_repo_clone'),
    Path.cwd(),
]

# Local Windows path should only be checked on Windows hosts.
if sys.platform.startswith('win'):
    FSAKE_DIR_CANDIDATES.append(Path('E:/projects/few_shot_gnn/fsake_repo_clone'))


def discover_fsake_dir(candidates):
    checked = []
    for p in candidates:
        p = p.resolve()
        checked.append(str(p))
        if (p / 'eval.py').exists() and (p / 'train.py').exists():
            return p, checked

    # Fallback: recursive search under common Colab roots
    search_roots = [Path('/content'), Path.cwd()]
    for root in search_roots:
        if not root.exists():
            continue
        for eval_py in root.rglob('eval.py'):
            parent = eval_py.parent
            if (parent / 'train.py').exists():
                checked.append(str(parent.resolve()))
                return parent.resolve(), checked

    return None, checked


def ensure_fsake_repo():
    """Ensure FSAKE repo exists in Colab, mirroring fsake_colab_guide clone flow."""
    fsake_dir, checked_paths = discover_fsake_dir(FSAKE_DIR_CANDIDATES)
    if fsake_dir is not None:
        return fsake_dir

    print('FSAKE repo not found. Checked paths:')
    for p in checked_paths:
        print(' -', p)

    if Path('/content').exists():
        print('\nAttempting Colab clone fallback...')
        repo_root = Path('/content/few_shot_article')
        src = repo_root / 'fsake_repo_clone'
        dst = Path('/content/FSAKE')

        if not repo_root.exists():
            subprocess.run(
                [
                    'git', 'clone', '--recursive',
                    'https://github.com/alirezakavianifar/few_shot_article.git',
                    str(repo_root),
                ],
                check=True,
            )

        if src.exists() and (src / 'eval.py').exists() and (src / 'train.py').exists():
            if dst.exists():
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f'Copied {src} -> {dst}')
            return dst

    raise RuntimeError('Could not find or create FSAKE repo with eval.py/train.py.')


fsake_dir = ensure_fsake_repo()
print('Using FSAKE dir:', fsake_dir)

# Ensure dataset exists at FSAKE expected relative path: ./dataset
# (eval.py uses dataset_root='dataset').
def ensure_fsake_dataset(fsake_dir):
    expected = fsake_dir / 'dataset' / 'mini-imagenet' / 'compacted_datasets' / 'mini_imagenet_test.pickle'
    if expected.exists():
        print('Dataset OK at:', expected)
        return

    dataset_root = fsake_dir / 'dataset'
    dataset_root.mkdir(parents=True, exist_ok=True)

    # Candidate dataset roots from common Colab layouts and this audit notebook.
    candidates = [
        Path('/content/FSAKE/dataset'),
        Path('/content/few_shot_article/fsake_repo_clone/dataset'),
        Path('/content/drive/MyDrive/FSAKE/dataset'),
        Path('/content/few_shot_gnn/FSAKE/dataset'),
        Path('/content/few_shot_gnn/fsake_repo_clone/dataset'),
        Path('/content/dataset'),
        Path.cwd() / 'dataset',
    ]

    src_dataset = None
    for c in candidates:
        probe = c / 'mini-imagenet' / 'compacted_datasets' / 'mini_imagenet_test.pickle'
        if probe.exists():
            src_dataset = c
            break

    if src_dataset is None:
        print('Could not find mini-imagenet test pickle in candidate dataset roots:')
        for c in candidates:
            print(' -', c)

        # Guide-aligned fallback: auto-download miniImageNet pickles via gdown
        # and place them under /content/FSAKE/dataset/mini-imagenet/compacted_datasets
        print('\nAttempting miniImageNet auto-download fallback (gdown)...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
        import gdown
        import os
        import tarfile
        import zipfile

        def _extract(archive_path, extract_to):
            os.makedirs(extract_to, exist_ok=True)
            with open(archive_path, 'rb') as fh:
                magic = fh.read(4)
            if magic[:2] == b'PK':
                with zipfile.ZipFile(archive_path) as z:
                    z.extractall(extract_to)
            else:
                try:
                    with tarfile.open(archive_path) as t:
                        t.extractall(extract_to, filter='data')
                except Exception:
                    with zipfile.ZipFile(archive_path) as z:
                        z.extractall(extract_to)

        def _gdown_try(file_id, out_path):
            approaches = [
                lambda: gdown.download(f'https://drive.google.com/file/d/{file_id}/view', out_path, quiet=False, fuzzy=True),
                lambda: gdown.download(id=file_id, output=out_path, quiet=False),
                lambda: gdown.download(id=file_id, output=out_path, quiet=False, use_cookies=False),
                lambda: gdown.download(f'https://drive.google.com/uc?export=download&id={file_id}', out_path, quiet=False),
            ]
            last_err = None
            for fn in approaches:
                try:
                    fn()
                    return
                except Exception as e:
                    last_err = e
                    if os.path.exists(out_path):
                        os.remove(out_path)
            raise RuntimeError(f'gdown failed for file_id={file_id}: {last_err}')

        archive_path = '/tmp/miniimagenet.archive'
        raw_root = '/tmp/mini_raw'
        file_id = '12V7qi-AjrYi6OoJdYcN_k502BM_jcP8D'
        if not os.path.exists(archive_path):
            _gdown_try(file_id, archive_path)
        _extract(archive_path, raw_root)

        mini_dst = fsake_dir / 'dataset' / 'mini-imagenet' / 'compacted_datasets'
        mini_dst.mkdir(parents=True, exist_ok=True)

        # Collect all pickle candidates once.
        pickle_candidates = []
        for r, _, files in os.walk(raw_root):
            for f in files:
                if f.endswith('.pickle'):
                    pickle_candidates.append(os.path.join(r, f))

        def pick_split(split):
            # 1) Prefer exact canonical names.
            canonical = [
                f'mini_imagenet_{split}.pickle',
                f'{split}.pickle',
            ]
            for c in canonical:
                for p in pickle_candidates:
                    if os.path.basename(p).lower() == c.lower():
                        return p

            # 2) Fallback to strict tokenized match (avoid accidental train->val matches).
            import re
            pat = re.compile(rf'(^|[^a-z]){split}([^a-z]|$)', re.IGNORECASE)
            matches = [p for p in pickle_candidates if pat.search(os.path.basename(p))]
            if len(matches) == 1:
                return matches[0]
            if len(matches) > 1:
                # Prefer shortest basename (usually canonical one) if ambiguous.
                matches = sorted(matches, key=lambda x: (len(os.path.basename(x)), os.path.basename(x)))
                return matches[0]
            return None

        selected = {}
        for split in ['train', 'val', 'test']:
            found = pick_split(split)
            if not found:
                raise RuntimeError(
                    f'Could not locate {split} pickle in downloaded miniImageNet archive. '
                    f'Candidates: {[os.path.basename(p) for p in pickle_candidates[:20]]}'
                )
            selected[split] = found
            shutil.copy(found, mini_dst / f'mini_imagenet_{split}.pickle')
            print(f'Placed mini_imagenet_{split}.pickle <- {os.path.basename(found)}')

        # Sanity validation: train/val/test must not be byte-identical.
        def _sha256_local(path):
            h = hashlib.sha256()
            with open(path, 'rb') as f:
                for chunk in iter(lambda: f.read(1024 * 1024), b''):
                    h.update(chunk)
            return h.hexdigest()

        t_sha = _sha256_local(mini_dst / 'mini_imagenet_train.pickle')
        v_sha = _sha256_local(mini_dst / 'mini_imagenet_val.pickle')
        te_sha = _sha256_local(mini_dst / 'mini_imagenet_test.pickle')
        if len({t_sha, v_sha, te_sha}) < 3:
            raise RuntimeError(
                'miniImageNet split extraction produced duplicate files (train/val/test hash collision). '
                'Run the dataset Option B cell from fsake_colab_guide and verify split pickles.'
            )

        src_dataset = fsake_dir / 'dataset'

    # Prefer symlink (cheap). Fallback to copytree if symlink is not allowed.
    target = fsake_dir / 'dataset'
    if target.exists():
        try:
            if target.is_symlink() or target.is_file():
                target.unlink()
        except Exception:
            pass

    # If source is already the target, do nothing
    if Path(src_dataset).resolve() == target.resolve():
        print('Dataset already placed at target:', target)
    else:
        try:
            if target.exists() and any(target.iterdir()):
                print('Dataset target already populated:', target)
            else:
                if target.exists():
                    shutil.rmtree(target)
                target.symlink_to(src_dataset, target_is_directory=True)
                print(f'Symlinked dataset: {src_dataset} -> {target}')
        except Exception:
            if target.exists():
                shutil.rmtree(target)
            shutil.copytree(src_dataset, target)
            print(f'Copied dataset: {src_dataset} -> {target}')

    if not expected.exists():
        raise RuntimeError(f'Dataset setup failed. Expected file still missing: {expected}')


ensure_fsake_dataset(fsake_dir)

asset_ckpt_root = fsake_dir / 'asset' / 'checkpoints'
asset_ckpt_root.mkdir(parents=True, exist_ok=True)

# Ensure downloaded checkpoints are mirrored into FSAKE expected path.
for exp in EXPERIMENTS:
    exp_name = exp['exp_name']
    exp_dst = asset_ckpt_root / exp_name
    exp_dst.mkdir(parents=True, exist_ok=True)

    for filename in ['checkpoint.pth.tar', 'model_best.pth.tar']:
        src = Path('./hf_checkpoint_audit') / exp_name / filename
        if src.exists():
            dst = exp_dst / filename
            shutil.copy2(src, dst)
            print(f'Copied {src} -> {dst}')
        else:
            print(f'Missing local file for mirror: {src}')

results = []
for exp in EXPERIMENTS:
    label = exp['label']
    exp_name = exp['exp_name']
    cmd = [sys.executable, 'eval.py'] + exp['eval_args']
    print('\n' + '=' * 80)
    print('Running:', label)
    print('Command:', ' '.join(cmd))

    proc = subprocess.run(cmd, cwd=str(fsake_dir), capture_output=True, text=True)
    combined = proc.stdout + '\n' + proc.stderr
    print(combined[-5000:])

    m = ACC_RE.search(combined)
    mean = float(m.group(1)) if m else None
    std = float(m.group(2)) if m else None
    ci95 = float(m.group(3)) if m else None

    results.append({
        'label': label,
        'exp_name': exp_name,
        'return_code': proc.returncode,
        'mean_acc': mean,
        'std': std,
        'ci95': ci95,
    })

print('\n=== EVAL SUMMARY ===')
for r in results:
    print(f"{r['label']:8} | rc={r['return_code']} | mean={r['mean_acc']} | std={r['std']} | ci95={r['ci95']}")

# Convenience delta if both succeeded
base = next((r for r in results if r['label'] == 'baseline' and r['mean_acc'] is not None), None)
lmt = next((r for r in results if r['label'] == 'lmt' and r['mean_acc'] is not None), None)
if base and lmt:
    print(f"\nDelta (LMT - Baseline): {lmt['mean_acc'] - base['mean_acc']:+.2f} percentage points")

In [ ]:
import os
import copy
import sys
import numpy as np
import torch
import importlib.util


def _load_module(module_name, file_path):
    # torchtools parses sys.argv on import and fails on Jupyter's '-f <json>' args.
    # Import with a clean argv, then restore.
    old_argv = list(sys.argv)
    try:
        sys.argv = [old_argv[0]]
        spec = importlib.util.spec_from_file_location(module_name, str(file_path))
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        return mod
    finally:
        sys.argv = old_argv


def _resolve_mediator_heads(embed_dim, requested_heads):
    if requested_heads <= 0:
        requested_heads = 1
    if embed_dim % requested_heads == 0:
        return requested_heads
    for h in range(requested_heads + 1, embed_dim + 1):
        if embed_dim % h == 0:
            return h
    for h in range(requested_heads - 1, 0, -1):
        if embed_dim % h == 0:
            return h
    return 1


def _set_tt_args_for_eval(tt, exp, fsake_root_dir):
    # Reset and set only args we need (mirrors eval.py defaults)
    tt.arg.device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    tt.arg.dataset_root = str(fsake_root_dir / 'dataset')
    tt.arg.dataset = 'mini'
    tt.arg.num_ways = 5
    tt.arg.num_shots = 1
    tt.arg.num_queries = tt.arg.num_ways * tt.arg.num_shots
    tt.arg.num_supports = tt.arg.num_ways * tt.arg.num_shots
    tt.arg.transductive = True
    tt.arg.meta_batch_size = 40
    tt.arg.seed = 222
    tt.arg.num_gpus = 1

    tt.arg.emb_size = 128
    tt.arg.in_dim = tt.arg.emb_size + tt.arg.num_ways

    tt.arg.pool_mode = 'support'
    tt.arg.unet_mode = 'addold'
    tt.arg.interaction_block = 'baseline'
    tt.arg.mediator_tokens = 8
    tt.arg.mediator_layers = 2
    tt.arg.mediator_heads = 4
    tt.arg.mediator_dropout = 0.1
    tt.arg.weight_decay = 0.0 # Added to prevent TypeError in Adam optimizer

    # Parse experiment-specific flags from configured eval_args
    args = exp['eval_args']
    def _get_flag(flag, default=None):
        if flag in args:
            idx = args.index(flag)
            if idx + 1 < len(args):
                return args[idx + 1]
        return default

    ib = _get_flag('--interaction_block', 'baseline')
    tt.arg.interaction_block = str(ib).lower()

    if tt.arg.interaction_block == 'lmt':
        tt.arg.mediator_tokens = int(_get_flag('--mediator_tokens', '8'))
        tt.arg.mediator_layers = int(_get_flag('--mediator_layers', '2'))
        tt.arg.mediator_heads = int(_get_flag('--mediator_heads', '4'))
        tt.arg.mediator_dropout = float(_get_flag('--mediator_dropout', '0.1'))

    lmt_feature_dim = tt.arg.in_dim - tt.arg.num_ways
    tt.arg.mediator_heads = _resolve_mediator_heads(lmt_feature_dim, tt.arg.mediator_heads)

    # Match eval episode count from eval.py
    tt.arg.test_iteration = 10000
    tt.arg.test_batch_size = 10


def _build_modules(model_mod, tt):
    enc_module = model_mod.EmbeddingImagenet(emb_size=tt.arg.emb_size)

    if tt.arg.interaction_block == 'lmt':
        unet_module = model_mod.LatentMediatorUnet(
            in_dim=tt.arg.in_dim,
            num_classes=tt.arg.num_ways,
            num_queries=tt.arg.num_queries,
            mediator_tokens=tt.arg.mediator_tokens,
            mediator_layers=tt.arg.mediator_layers,
            mediator_heads=tt.arg.mediator_heads,
            mediator_dropout=tt.arg.mediator_dropout,
        )
    else:
        # For this notebook we evaluate mini 5-way 1-shot transductive support mode
        ks = [0.6, 0.5]
        unet_module = model_mod.Unet(ks, tt.arg.in_dim, tt.arg.num_ways, tt.arg.num_queries)

    return enc_module, unet_module


# Locate repo prepared by previous cell
if 'fsake_dir' not in globals() or fsake_dir is None:
    raise RuntimeError('Run the previous evaluation setup cell first (it defines fsake_dir).')

if str(fsake_dir) not in sys.path:
    sys.path.insert(0, str(fsake_dir))

# Import repo modules directly from fsake_dir
train_mod = _load_module('fsake_train_mod', fsake_dir / 'train.py')
data_mod = _load_module('fsake_data_mod', fsake_dir / 'data.py')
model_mod = _load_module('fsake_model_mod', fsake_dir / 'model.py')

# Use torchtools tt object from train module namespace
tt = train_mod.tt

consistency_rows = []
for exp in EXPERIMENTS:
    exp_name = exp['exp_name']
    label = exp['label']

    ckpt_path = fsake_dir / 'asset' / 'checkpoints' / exp_name / 'model_best.pth.tar'
    if not ckpt_path.exists():
        print(f'[{label}] missing checkpoint: {ckpt_path}')
        continue

    _set_tt_args_for_eval(tt, exp, fsake_dir)

    # Build loaders (val can be absent in some prepared datasets)
    val_pickle = fsake_dir / 'dataset' / 'mini-imagenet' / 'compacted_datasets' / 'mini_imagenet_val.pickle'
    test_pickle = fsake_dir / 'dataset' / 'mini-imagenet' / 'compacted_datasets' / 'mini_imagenet_test.pickle'

    if not test_pickle.exists():
        raise RuntimeError(f'Missing required test pickle: {test_pickle}')

    loaders = {'test': data_mod.MiniImagenetLoader(root=tt.arg.dataset_root, partition='test')}
    has_val = val_pickle.exists()
    if has_val:
        loaders['val'] = data_mod.MiniImagenetLoader(root=tt.arg.dataset_root, partition='val')
    else:
        print(f"[{label}] mini_imagenet_val.pickle not found; skipping val evaluation.")

    # Build model and trainer
    enc_module, unet_module = _build_modules(model_mod, tt)
    trainer = train_mod.ModelTrainer(
        enc_module=enc_module,
        unet_module=unet_module,
        data_loader=loaders,
    )

    # Load model_best state
    state = torch.load(ckpt_path, map_location=tt.arg.device, weights_only=False)
    trainer.enc_module.load_state_dict(state['enc_module_state_dict'])
    trainer.unet_module.load_state_dict(state['unet_module_state_dict'])
    trainer.global_step = int(state.get('iteration', 0))
    trainer.val_acc = float(state.get('best_val_acc', state.get('val_acc', 0.0)))

    # Evaluate with same runtime/protocol
    val_now = float(trainer.eval(partition='val')) if has_val else np.nan
    test_now = float(trainer.eval(partition='test'))

    row = {
        'label': label,
        'exp_name': exp_name,
        'ckpt_iteration': int(state.get('iteration', -1)),
        'ckpt_val_acc': float(state.get('val_acc', np.nan)),
        'ckpt_best_val_acc': float(state.get('best_val_acc', np.nan)),
        'eval_val_now': val_now,
        'eval_test_now': test_now,
        'val_test_gap_pp': ((val_now - test_now) * 100.0) if has_val else np.nan,
    }
    consistency_rows.append(row)

print('\n=== CONSISTENCY CHECK (model_best) ===')
for r in consistency_rows:
    print('-' * 100)
    print(f"[{r['label']}] {r['exp_name']}")
    print(f"  ckpt iteration      : {r['ckpt_iteration']}")
    print(f"  ckpt val_acc        : {r['ckpt_val_acc']:.6f}")
    print(f"  ckpt best_val_acc   : {r['ckpt_best_val_acc']:.6f}")
    if np.isnan(r['eval_val_now']):
        print('  eval val now        : N/A (mini_imagenet_val.pickle missing)')
        print(f"  eval test now       : {r['eval_test_now']:.6f} ({r['eval_test_now']*100:.2f}%)")
        print('  val-test gap        : N/A')
    else:
        print(f"  eval val now        : {r['eval_val_now']:.6f} ({r['eval_val_now']*100:.2f}%)")
        print(f"  eval test now       : {r['eval_test_now']:.6f} ({r['eval_test_now']*100:.2f}%)")
        print(f"  val-test gap        : {r['val_test_gap_pp']:.2f} pp")

In [ ]:
# Dataset split integrity audit (miniImageNet pickles)
# Verifies train/val/test pickle content, class counts, image counts, and hashes.

import os
import pickle
import hashlib
from collections import Counter


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def load_pickle_robust(path):
    try:
        with open(path, 'rb') as f:
            return pickle.load(f)
    except Exception:
        with open(path, 'rb') as f:
            return pickle.load(f, encoding='latin1')


def summarize_split(path):
    obj = load_pickle_robust(path)

    # Supported formats:
    # 1) dict[label] -> list(images)
    # 2) dict with keys: catname2label, labels, data
    result = {
        'path': str(path),
        'exists': os.path.exists(path),
        'size_mb': round(os.path.getsize(path) / (1024 * 1024), 2),
        'sha256': sha256_file(path),
        'format': None,
        'num_classes': None,
        'num_images': None,
        'top5_class_counts': None,
        'sample_class_ids': None,
    }

    if isinstance(obj, dict) and all(k in obj for k in ('catname2label', 'labels', 'data')):
        labels = obj['labels']
        c = Counter(labels)
        result['format'] = 'split-format(catname2label,labels,data)'
        result['num_classes'] = len(c)
        result['num_images'] = len(labels)
        result['top5_class_counts'] = c.most_common(5)
        result['sample_class_ids'] = list(sorted(c.keys()))[:10]
        return result

    if isinstance(obj, dict):
        # assume class->list(images)
        class_ids = list(obj.keys())
        counts = {}
        total = 0
        for k, v in obj.items():
            n = len(v) if hasattr(v, '__len__') else 0
            counts[k] = n
            total += n
        c = Counter(counts)
        result['format'] = 'class-dict(label->list[images])'
        result['num_classes'] = len(class_ids)
        result['num_images'] = total
        result['top5_class_counts'] = c.most_common(5)
        # normalize printable sample ids
        sample_ids = []
        for k in sorted(class_ids)[:10]:
            if isinstance(k, bytes):
                try:
                    sample_ids.append(k.decode('utf-8'))
                except Exception:
                    sample_ids.append(str(k))
            else:
                sample_ids.append(str(k))
        result['sample_class_ids'] = sample_ids
        return result

    result['format'] = f'unexpected:{type(obj)}'
    return result


if 'fsake_dir' not in globals() or fsake_dir is None:
    raise RuntimeError('Run setup/eval cell first so fsake_dir is defined.')

mini_root = fsake_dir / 'dataset' / 'mini-imagenet' / 'compacted_datasets'
split_files = {
    'train': mini_root / 'mini_imagenet_train.pickle',
    'val': mini_root / 'mini_imagenet_val.pickle',
    'test': mini_root / 'mini_imagenet_test.pickle',
}

print('mini root:', mini_root)

summaries = {}
for split, path in split_files.items():
    print('\n' + '=' * 100)
    print(f'[{split}] {path}')
    if not path.exists():
        print('MISSING')
        summaries[split] = {'exists': False}
        continue

    s = summarize_split(path)
    summaries[split] = s

    print('exists           :', s['exists'])
    print('size_mb          :', s['size_mb'])
    print('sha256           :', s['sha256'])
    print('format           :', s['format'])
    print('num_classes      :', s['num_classes'])
    print('num_images       :', s['num_images'])
    print('top5_class_counts:', s['top5_class_counts'])
    print('sample_class_ids :', s['sample_class_ids'])

print('\n' + '=' * 100)
print('EXPECTED (common miniImageNet protocol): train~64 classes, val~16 classes, test~20 classes')
print('Observed:')
for split in ['train', 'val', 'test']:
    s = summaries.get(split, {})
    print(f"  {split:5} -> classes={s.get('num_classes')} images={s.get('num_images')}")

# Quick mismatch heuristic
obs = {k: summaries.get(k, {}).get('num_classes') for k in ['train', 'val', 'test']}
if all(v is not None for v in obs.values()):
    if obs != {'train': 64, 'val': 16, 'test': 20}:
        print('\n[ALERT] Split cardinalities differ from common miniImageNet protocol (64/16/20).')
        print('This strongly supports a split/protocol mismatch as the cause of the val-test gap.')
    else:
        print('\n[OK] Split class counts match common miniImageNet protocol (64/16/20).')